# Ames Housing - Step 1: Explore and Understand The Data

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import seaborn as sns
from IPython.display import display

current = Path.cwd()
project_candidates = [current, current / "IE500618Assignment1", *current.parents]
project_root = next(
    candidate
    for candidate in project_candidates
    if (candidate / "src" / "config.py").exists()
)
sys.path.insert(0, str(project_root / "src"))

from config import PATH_AMESHOUSING
from setup_plot_style import *

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)

## 1. Load and Inspect Data

In [ ]:
df = pd.read_csv(PATH_AMESHOUSING)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
print(f"Dataset path: {PATH_AMESHOUSING}")
display(df.head())

In [ ]:
df.info()

In [ ]:
# Overview of variables and data types
df_structure = pd.DataFrame({
    "data_type": df.dtypes.astype(str),
    "non_missing": df.notna().sum(),
    "unique_values": df.nunique(dropna=False),
})

print(f"Numeric variables: {df.select_dtypes(include="number").shape[1]}")
print(f"Categorical variables: {df.select_dtypes(exclude="number").shape[1]}")
display(df_structure)

df.info()

## 2. Investigate Missing Values

In [ ]:
df_missing = (
    df.isna()
      .sum()
      .rename("missing_count")
      .to_frame()
      .assign(missing_percent=lambda x: 100 * x["missing_count"] / len(df))
      .query("missing_count > 0")
      .sort_values("missing_count", ascending=False)
)

df_missing = df_missing.round(2)
display(df_missing)
print(f"Columns with at least one missing value: {len(df_missing)}")

In [ ]:
# Original horizontal version
setup_plot_style(14, 8)
ax = sns.barplot(
    data=df_missing,
    x="missing_percent",
    y=df_missing.index,
    alpha=1,
    edgecolor="black"
)
ax.xaxis.set_major_formatter(mtick.StrMethodFormatter("{x:,.0f}%"))
ax.set_title("Columns with The Missing Values", fontweight="bold")
ax.set_xlabel("Missing values (%)")
ax.set_ylabel("Column")
ax.bar_label(ax.containers[0], fmt="%.2f%%", padding=3)

plt.xlim(0, 105)
plt.tight_layout()
plt.show()

# Visualise the columns
top_missing = df_missing.sort_values(
    "missing_percent", ascending=False)

setup_plot_style(14, 8)
ax = sns.barplot(
    data=top_missing,
    x=top_missing.index,
    y="missing_percent",
    # color="#4C78A8",
    alpha=1,
    edgecolor="black"
)

plt.gca().yaxis.set_major_formatter(
    mtick.StrMethodFormatter('{x:,.0f}%')
)

ax.set_title("Columns with The Missing Values", fontweight="bold")
ax.set_xlabel("Column")
ax.set_ylabel("Missing Values (%)")
ax.bar_label(
    ax.containers[0],
    fmt="%.2f%%",
    padding=3,
    rotation=0,
    fontsize=8,
)

#plt.yscale("log")
plt.xticks(rotation=60, ha="right")
plt.tight_layout()
plt.show()

# Total number of missing values
setup_plot_style(14, 8)
ax = sns.barplot(
    data=top_missing,
    x=top_missing.index,
    y="missing_count",
    alpha=1,
    edgecolor="black"
)
ax.set_title("Total Missing Values by Column", fontweight="bold")
ax.set_xlabel("Column")
ax.set_ylabel("Amount of Missing Values")
ax.bar_label(ax.containers[0], padding=3, fontsize=8)

#plt.yscale("log")
plt.xticks(rotation=60, ha="right")
plt.tight_layout()
plt.show()

## 3. Explore `SalePrice`

In [ ]:
sale_price = df["SalePrice"]

sale_price_summary = sale_price.describe().to_frame().T.round(2)
display(sale_price_summary)

In [ ]:
sale_price_summary = (
    df["SalePrice"]
    .describe()
    .to_frame()
    .T
    .round(2)
)

display(sale_price_summary)

In [ ]:
setup_plot_style()

sns.histplot(
    sale_price,
    bins=100,
    edgecolor="black",
    alpha=1,
)

plt.gca().xaxis.set_major_formatter(
    mtick.StrMethodFormatter('${x:,.0f}')
)

plt.title("Distribution of Sale Prices")
plt.xlabel("Sale Price (USD)")
plt.ylabel("Amount")
plt.show()

In [ ]:
median = sale_price.median()

setup_plot_style(12, 4)

sns.boxplot(
    x=sale_price,
    width=0.15,
    saturation=1,
    flierprops={
        "markersize": 2.5,
        "markerfacecolor": "red",
        "markeredgecolor": "black",
        "alpha": 0.25
    },
    boxprops={"edgecolor": "black"}
)

plt.gca().xaxis.set_major_formatter(
    mtick.StrMethodFormatter('${x:,.0f}')
)

plt.text(
    median, 0.2,
    f"Median: ${median:,.0f}",
    ha="center"
)

plt.title("Sale Price", fontweight="bold")
plt.xlabel("Sale Price (USD)")
plt.show()

In [ ]:
# Most expensive houses
display(
    df[["SalePrice", "Gr Liv Area", "Overall Qual", "Year Built", "Neighborhood"]]
    .sort_values("SalePrice", ascending=False)
    .head(10)
)

## 4. Looking for Numerical Relationships

In [ ]:
# Heatmap for correlation
plt.figure(figsize=(40, 30))
sns.heatmap(
    data=df.corr(numeric_only=True),
    annot=True,
    fmt=".2f",
    cmap="Spectral",
    center=0,
    linewidths=0.5,
    linecolor="black"
)


plt.title("Correlation of the whole dataframe", fontweight="bold")
plt.show()

In [ ]:
numeric_df = df.select_dtypes(include="number").drop(
    columns=["Order", "PID"], errors="ignore")
sale_price_correlations = (
    numeric_df.corr(numeric_only=True)["SalePrice"]
    .drop("SalePrice")
    .sort_values(key=abs, ascending=False)
)

display(sale_price_correlations.head(15).round(
    3).to_frame("SalePrice_corr"))

In [ ]:
numeric_df = df.select_dtypes(include="number").drop(
    columns=["Order", "PID"], errors="ignore")
overall_quality_correlations = (
    numeric_df.corr(numeric_only=True)["Overall Qual"]
    .drop("Overall Qual", errors="ignore")
    .sort_values(key=abs, ascending=False)
)

display(overall_quality_correlations.head(15).round(
    3).to_frame("Overall_Qual_corr"))

In [ ]:
# Heatmap of Sale Price correlations
df_sale_price_correlations = sale_price_correlations.abs().head(
    10).index.tolist() + ["SalePrice"]

setup_plot_style(8, 7)
sns.heatmap(
    df[df_sale_price_correlations].corr(),
    annot=True,
    fmt=".2f",
    cmap="Spectral",
    center=0,
    linewidths=0.5,
    linecolor="black"
)
plt.title("Correlations Among Sale Price", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap of Overall Qual correlations
top_numeric_features = overall_quality_correlations.abs().head(
    10).index.tolist() + ["Overall Qual"]

setup_plot_style(8, 7)
sns.heatmap(
    df[top_numeric_features].corr(),
    annot=True,
    fmt=".2f",
    cmap="Spectral",
    center=0,
    linewidths=0.5,
    linecolor="black"
)
plt.title("Correlations with Overall Quality", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Number of houses by sale price and overall quality

setup_plot_style()

sns.histplot(
    data=df,
    x="SalePrice",
    hue="Overall Qual",
    bins=100,
    multiple="stack",
    edgecolor="black",
    alpha=1,
    palette="viridis"
)

plt.gca().xaxis.set_major_formatter(
    mtick.StrMethodFormatter("${x:,.0f}")
)

plt.title("Distribution of Sale Prices by Overall Quality", fontweight="bold")
plt.xlabel("Sale Price (USD)")
plt.ylabel("Amount")
plt.show()

In [ ]:
quality_counts = (
    df["Overall Qual"]
    .value_counts()
    .sort_index()
    .rename_axis("Overall Qual")
    .reset_index(name="Amount")
)

quality_table = quality_counts.rename(
    columns={
        "Overall Qual": "Overall Quality",
        "Amount": "Number of Houses"
    }
)

display(quality_table)

In [ ]:
# Number of houses by overall quality

setup_plot_style()

ax = sns.barplot(
    data=quality_counts,
    x="Overall Qual",
    y="Amount",
    alpha=1,
    edgecolor="black"
)

for container in ax.containers:
    ax.bar_label(container, fmt="%.0f", padding=3, rotation=0, fontsize=8)

ax.set_title(
    "Number of Houses by Overall Quality",
    fontweight="bold"
)
ax.set_xlabel("Overall Quality")
ax.set_ylabel("Amount")

plt.tight_layout()
plt.show()

In [ ]:
quality_summary = pd.DataFrame({
    "Mean": [df["Overall Qual"].mean()],
    "Median": [df["Overall Qual"].median()],
    "Most common": [df["Overall Qual"].mode()[0]],
    "Minimum": [df["Overall Qual"].min()],
    "Maximum": [df["Overall Qual"].max()]
})

display(quality_summary.round(2))

## 5. Plots of Important Relationships

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.scatterplot(
    data=df,
    x="Gr Liv Area",
    y="SalePrice",
    alpha=1,
    edgecolor="black",
    linewidth=0.25,
    ax=axes[0, 0]
)
axes[0, 0].set_title("Living Area and Sale Price")

sns.boxplot(
    data=df,
    x="Overall Qual",
    y="SalePrice",
    width=0.5,
    saturation=1,
    flierprops={
        "markersize": 2.5,
        "markerfacecolor": "red",
        "markeredgecolor": "black",
        "alpha": 0.25
    },
    boxprops={"edgecolor": "black"},
    ax=axes[0, 1]
)
axes[0, 1].set_title("Overall Quality and Sale Price")

sns.scatterplot(
    data=df,
    x="Year Built",
    y="SalePrice",
    alpha=1,
    edgecolor="black",
    linewidth=0.25,
    ax=axes[1, 0]
)
axes[1, 0].set_title("Year Built and Sale Price")

sns.scatterplot(
    data=df,
    x="Garage Area",
    y="SalePrice",
    alpha=1,
    edgecolor="black",
    linewidth=0.25,
    ax=axes[1, 1]
)
axes[1, 1].set_title("Garage Area and Sale Price")

for ax in axes.flat:
    ax.set_ylabel("Sale Price (USD)")
    ax.yaxis.set_major_formatter(
        mtick.StrMethodFormatter('${x:,.0f}')
    )
    ax.title.set_fontweight("bold")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.scatterplot(
    data=df,
    x="Gr Liv Area",
    y="SalePrice",
    hue="Overall Qual",
    palette="viridis",
    alpha=0.8,
    edgecolor="black",
    linewidth=0.25,
    ax=axes[0, 0]
)
axes[0, 0].set_title("Living Area and Sale Price")

sns.boxplot(
    data=df,
    x="Overall Qual",
    y="SalePrice",
    width=0.5,
    saturation=1,
    palette="viridis",
    flierprops={
        "markersize": 2.5,
        "markerfacecolor": "red",
        "markeredgecolor": "black",
        "alpha": 0.25
    },
    boxprops={"edgecolor": "black"},
    ax=axes[0, 1]
)
axes[0, 1].set_title("Overall Quality and Sale Price")

sns.scatterplot(
    data=df,
    x="Year Built",
    y="SalePrice",
    hue="Overall Qual",
    palette="viridis",
    alpha=0.8,
    edgecolor="black",
    linewidth=0.25,
    ax=axes[1, 0]
)
axes[1, 0].set_title("Year Built and Sale Price")

sns.scatterplot(
    data=df,
    x="Garage Area",
    y="SalePrice",
    hue="Overall Qual",
    palette="viridis",
    alpha=0.8,
    edgecolor="black",
    linewidth=0.25,
    ax=axes[1, 1]
)
axes[1, 1].set_title("Garage Area and Sale Price")

for ax in axes.flat:
    ax.set_ylabel("Sale Price (USD)")
    ax.yaxis.set_major_formatter(
        mtick.StrMethodFormatter('${x:,.0f}')
    )
    ax.title.set_fontweight("bold")
    legend = ax.get_legend()
    if legend:
        legend.set_title("Overall Qual", prop={"size": 10})
        for text in legend.get_texts():
            text.set_fontsize(8)

plt.tight_layout()
plt.show()

In [ ]:
# Orders
order_by_price = (
    df.groupby("Neighborhood")["SalePrice"]
      .median()
      .sort_values()
      .index
)

order_by_name = sorted(
    df["Neighborhood"]
    .dropna()
    .unique()
)


# Sorted by sale price
plt.figure(figsize=(14, 6))

sns.boxplot(
    data=df,
    x="Neighborhood",
    y="SalePrice",
    order=order_by_price,
    hue="Neighborhood",
    palette="Spectral",
    legend=False,
    width=0.6,
    saturation=1,
    flierprops={
        "markersize": 2.5,
        "markerfacecolor": "red",
        "markeredgecolor": "black",
        "alpha": 0.25
    },
    boxprops={"edgecolor": "black"}
)

plt.gca().yaxis.set_major_formatter(
    mtick.StrMethodFormatter('${x:,.0f}')
)

plt.xticks(rotation=65, ha="right")
plt.title("Sale Price Distributions by Neighbourhood - Sorted by Price",
          fontweight="bold")
plt.xlabel("Neighbourhood")
plt.ylabel("Sale Price (USD)")
plt.tight_layout()
plt.show()


# Sorted alphabetically
plt.figure(figsize=(14, 6))

sns.boxplot(
    data=df,
    x="Neighborhood",
    y="SalePrice",
    order=order_by_name,
    hue="Neighborhood",
    palette="Spectral",
    legend=False,
    width=0.6,
    saturation=1,
    flierprops={
        "markersize": 2.5,
        "markerfacecolor": "red",
        "markeredgecolor": "black",
        "alpha": 0.25
    },
    boxprops={"edgecolor": "black"}
)

plt.gca().yaxis.set_major_formatter(
    mtick.StrMethodFormatter('${x:,.0f}')
)

plt.xticks(rotation=65, ha="right")
plt.title("Sale Price Distributions by Neighbourhood - Sorted by Name",
          fontweight="bold")
plt.xlabel("Neighbourhood")
plt.ylabel("Sale Price (USD)")
plt.tight_layout()
plt.show()